# Yelp — XGBoost with Learned Embeddings
**Features**: 6 numerical + state(4d) + city(8d) + categories(8d) + user(16d) + business(16d) = ~58 dense features
**Reads from**: `embeddings/`
**Saves to**: `embeddings_XGB/`

**Key difference from baseline**: ~58 dense features vs ~1,944 sparse features.
RF can now use full default hyperparameters.


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Check embedding files
STEP 3 — Load data
STEP 4 — Load embeddings
STEP 5 — Build dense feature matrix
STEP 6 — Train & evaluate
STEP 7 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error, mean_absolute_error

DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings_XGB"
os.makedirs(OUT_DIR, exist_ok=True)


---
## Step 2 — Check Embedding Files

In [ ]:
required = ['state_embeddings.npy','city_embeddings.npy','cat_embeddings.npy',
            'user_embeddings.npy','business_embeddings.npy',
            'state_encoder.json','city_encoder.json','cat_encoder.json',
            'user_encoder.json','business_encoder.json']
for fname in required:
    fpath  = os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings", fname)
    status = '✓' if os.path.exists(fpath) else '✗ MISSING — run EmbeddingANN first'
    print(f"  {status}  {fname}")


---
## Step 3 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
NUM_COLS  = [
    'user_avg_rating',    'user_rating_count',
    'business_avg_rating','business_rating_count', 'business_rating_std',
    'days_since_review'
]
TARGET    = 'stars'
CAT_STATE = 'state'
CAT_CITY  = 'city'
CAT_CATS  = 'categories'


---
## Step 4 — Load Embeddings

In [ ]:
EMB_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings"
# Load weight matrices
state_w = np.load(os.path.join(EMB_DIR, 'state_embeddings.npy'))
city_w  = np.load(os.path.join(EMB_DIR, 'city_embeddings.npy'))
cat_w   = np.load(os.path.join(EMB_DIR, 'cat_embeddings.npy'))
user_w  = np.load(os.path.join(EMB_DIR, 'user_embeddings.npy'))
biz_w   = np.load(os.path.join(EMB_DIR, 'business_embeddings.npy'))

# Load encoders
with open(os.path.join(EMB_DIR, 'state_encoder.json'))    as f: state2idx = json.load(f)
with open(os.path.join(EMB_DIR, 'city_encoder.json'))     as f: city2idx  = json.load(f)
with open(os.path.join(EMB_DIR, 'cat_encoder.json'))      as f: cat2idx   = json.load(f)
with open(os.path.join(EMB_DIR, 'user_encoder.json'))     as f: user2idx  = json.load(f)
with open(os.path.join(EMB_DIR, 'business_encoder.json')) as f: biz2idx   = json.load(f)

print(f"State  : {state_w.shape}  City : {city_w.shape}  Cat : {cat_w.shape}")
print(f"User   : {user_w.shape}   Biz  : {biz_w.shape}")


---
## Step 5 — Build Dense Feature Matrix

In [ ]:
def lookup(series, encoder, weights):
    idx = series.map(lambda x: encoder.get(str(x), 0)).values
    return weights[idx]

def mean_pool_cats(series, cat2idx, cat_w):
    result = []
    for c_str in series.fillna('<UNK>'):
        tokens  = c_str.split('|')
        indices = [cat2idx.get(t, 0) for t in tokens]
        vecs    = cat_w[indices]
        result.append(vecs.mean(axis=0))
    return np.array(result, dtype=np.float32)

def build_matrix(df_, num_arr):
    state_vecs = lookup(df_['state'],       state2idx, state_w)
    city_vecs  = lookup(df_['city'],        city2idx,  city_w)
    cat_vecs   = mean_pool_cats(df_['categories'], cat2idx, cat_w)
    user_vecs  = lookup(df_['user_id'],     user2idx,  user_w)
    biz_vecs   = lookup(df_['business_id'], biz2idx,   biz_w)
    return np.concatenate([num_arr, state_vecs, city_vecs,
                           cat_vecs, user_vecs, biz_vecs], axis=1)

scaler      = StandardScaler()
X_train_num = scaler.fit_transform(df_train[NUM_COLS])
X_val_num   = scaler.transform(df_val[NUM_COLS])
X_test_num  = scaler.transform(df_test[NUM_COLS])

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values

X_train = build_matrix(df_train, X_train_num)
X_val   = build_matrix(df_val,   X_val_num)
X_test  = build_matrix(df_test,  X_test_num)

total_dim = (X_train.shape[1])
print(f"Feature matrix — train: {X_train.shape}")
print(f"  6 num + {state_w.shape[1]} state + {city_w.shape[1]} city + "
      f"{cat_w.shape[1]} cat + {user_w.shape[1]} user + {biz_w.shape[1]} biz "
      f"= {total_dim} total")


---
## Step 6 — Train & Evaluate
XGBoost with embedding features.

In [ ]:
def get_metrics(y_true, y_pred):
    rmse = round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4)
    mae  = round(float(mean_absolute_error(y_true, y_pred)), 4)
    return rmse, mae


In [ ]:
t0    = time.perf_counter()
model = XGBRegressor(tree_method='hist', random_state=42, verbosity=0)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
train_time = time.perf_counter() - t0
print(f"Trained in {train_time:.1f}s")

In [ ]:
tr_rmse,  tr_mae  = get_metrics(y_train, model.predict(X_train))
val_rmse, val_mae = get_metrics(y_val,   model.predict(X_val))
te_rmse,  te_mae  = get_metrics(y_test,  model.predict(X_test))

print("=" * 55)
print(f"XGBoost [Embeddings] — Results")
print("=" * 55)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.2f}s")
print(f"  Train-Test gap : {round(te_rmse-tr_rmse,4)}")


---
## Step 7 — Save Results

In [ ]:
result = {
    'model'        : 'XGBoost [Embeddings]',
    'train_rmse'   : tr_rmse, 'val_rmse'  : val_rmse, 'test_rmse'  : te_rmse,
    'train_mae'    : tr_mae,  'val_mae'   : val_mae,  'test_mae'   : te_mae,
    'train_time_s' : round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings_XGB", 'xgb_results.csv'), index=False)
with open(os.path.join(r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings_XGB", 'xgb_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved → C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\embeddings_XGB")
print(f"Test RMSE: {te_rmse}  Test MAE: {te_mae}")
